# Eksperyment: Stabilność modelu — porównanie wersji bazowej (17 cech) i rozszerzonej (40 cech)

## Cel eksperymentu
Ocena **stabilności** i **powtarzalności** wyników dwóch wersji modelu predykcji wyników meczów tenisowych ATP. Każdy model uruchamiany jest **10 razy** z różnymi ziarnami losowości (RANDOM_STATE), a następnie wyniki są uśredniane.

Porównywane wersje:
1. **Model bazowy (17 cech)** — `main copy.py`: ranking, wiek, wzrost, forma, forma nawierzchniowa, H2H + cechy różnicowe. Historia: 2018–2023.
2. **Model rozszerzony (40 cech)** — `main.py`: jak wyżej + statystyki serwisowe (16 cech), best_of, runda, ręczność, punkty rankingowe. Historia: 2018–2023.

## Metodologia wielokrotnego uruchomienia

| Element | Stały / Zmienny | Opis |
|---------|-----------------|------|
| Dane wejściowe (2024) | Stały | Identyczne mecze w obu modelach |
| Podział chronologiczny (60/20/20) | Stały | Ten sam podział train/val/test |
| Cechy dynamiczne (forma, H2H, serwis) | Stałe | Obliczane raz — nie zależą od ziarna |
| **RANDOM_STATE** | **Zmienny** | Ziarna 1, 2, ..., 10 |
| RandomizedSearchCV | Zmienny | Losowanie hiperparametrów zależy od ziarna |
| Shuffle symetryzacji | Zmienny | Kolejność próbek treningowych zależy od ziarna |
| Random Forest | Zmienny | Wewnętrzny bagging zależy od ziarna |

## Raportowane metryki
- **CV Accuracy** — walidacja krzyżowa (TimeSeriesSplit, 5 foldów)
- **Validation Accuracy** — zbiór walidacyjny (symetryzowany)
- **Test Accuracy** — zbiór testowy (symetryzowany)
- **Match Accuracy** — trafność przewidywania zwycięzcy na poziomie meczów (główna metryka)

In [ ]:
import os
os.environ['PYTHONWARNINGS'] = 'ignore'

import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import time

N_RUNS = 10
SEEDS = list(range(1, N_RUNS + 1))  # Ziarna: 1, 2, ..., 10
print(f"Liczba uruchomień: {N_RUNS}, ziarna: {SEEDS}")

## 1. Wczytanie i przygotowanie danych (2024)

Wczytujemy dane z sezonu 2024 — to zbiór, na którym oceniany jest model. Definiujemy **dwie wersje** kolumn bazowych:
- `cols_base_17` — 10 kolumn dla modelu bazowego (17 cech)
- `cols_base_40` — 34 kolumny dla modelu rozszerzonego (16 bazowych + 18 serwisowych)

Dla modelu rozszerzonego dodatkowo obliczamy:
- Logarytm punktów rankingowych (`rank_pts_log`)
- Ręczność gracza (`is_lefty`)
- Numeryczny kod rundy turnieju (`round_encoded`)

In [ ]:
df = pd.read_csv('sample_data/2024.csv')
df['tourney_date'] = pd.to_datetime(df['tourney_date'], format='%Y%m%d')
df = df.sort_values(['tourney_date', 'match_num']).reset_index(drop=True)

# --- Kolumny bazowe: model 17-cechowy ---
cols_base_17 = ['surface', 'tourney_level', 'winner_rank', 'winner_age', 'winner_ht',
                'loser_rank', 'loser_age', 'loser_ht', 'winner_name', 'loser_name']

# --- Kolumny bazowe: model 40-cechowy (z serwisem, rundą, ręcznością, rank_points) ---
cols_serve = ['w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon',
              'w_SvGms', 'w_bpSaved', 'w_bpFaced',
              'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon', 'l_2ndWon',
              'l_SvGms', 'l_bpSaved', 'l_bpFaced']

cols_base_40 = ['surface', 'tourney_level', 'best_of', 'round',
                'winner_rank', 'winner_age', 'winner_ht', 'winner_hand', 'winner_rank_points',
                'loser_rank', 'loser_age', 'loser_ht', 'loser_hand', 'loser_rank_points',
                'winner_name', 'loser_name'] + cols_serve

# --- Model 17-cechowy ---
df_base_17 = df[cols_base_17].dropna().copy()
df_base_17['winner_rank_log'] = np.log(df_base_17['winner_rank'])
df_base_17['loser_rank_log'] = np.log(df_base_17['loser_rank'])

# --- Model 40-cechowy ---
df_base_40 = df[cols_base_40].dropna().copy()
df_base_40['winner_rank_log'] = np.log(df_base_40['winner_rank'])
df_base_40['loser_rank_log'] = np.log(df_base_40['loser_rank'])
df_base_40['winner_rank_pts_log'] = np.log(df_base_40['winner_rank_points'])
df_base_40['loser_rank_pts_log'] = np.log(df_base_40['loser_rank_points'])
df_base_40['winner_is_lefty'] = (df_base_40['winner_hand'] == 'L').astype(int)
df_base_40['loser_is_lefty'] = (df_base_40['loser_hand'] == 'L').astype(int)

ROUND_ORDER = {'R128': 1, 'R64': 2, 'R32': 3, 'RR': 3, 'R16': 4, 'QF': 5, 'SF': 6, 'BR': 6, 'F': 7}
df_base_40['round_encoded'] = df_base_40['round'].map(ROUND_ORDER).fillna(3)

print(f"Dane 2024 — model 17-cechowy: {len(df_base_17)} meczów")
print(f"Dane 2024 — model 40-cechowy: {len(df_base_40)} meczów")

## 2. Dane historyczne (redukcja cold-start)

Oba modele korzystają z historii **2018–2023** (6 sezonów), zgodnie z oryginalną implementacją w `main copytest.py` i `maintest.py`. Więcej historii oznacza dokładniejsze oszacowanie formy, H2H i statystyk serwisowych.

In [ ]:
# --- Historia dla modelu 17-cechowego (2018–2023, zgodnie z main copytest.py) ---
history_files_17 = ['sample_data/2018.csv', 'sample_data/2019.csv', 'sample_data/2020.csv',
                    'sample_data/2021.csv', 'sample_data/2022.csv', 'sample_data/2023.csv']
history_parts_17 = []
for filepath in history_files_17:
    try:
        df_hist = pd.read_csv(filepath)
        df_hist['tourney_date'] = pd.to_datetime(df_hist['tourney_date'], format='%Y%m%d')
        df_hist = df_hist.sort_values(['tourney_date', 'match_num']).reset_index(drop=True)
        history_parts_17.append(df_hist[cols_base_17].dropna().copy())
    except FileNotFoundError:
        pass
df_history_17 = pd.concat(history_parts_17, ignore_index=True) if history_parts_17 else pd.DataFrame(columns=cols_base_17)

# --- Historia dla modelu 40-cechowego (2018–2023) ---
history_files_40 = ['sample_data/2018.csv', 'sample_data/2019.csv', 'sample_data/2020.csv',
                    'sample_data/2021.csv', 'sample_data/2022.csv', 'sample_data/2023.csv']
history_parts_40 = []
for filepath in history_files_40:
    try:
        df_hist = pd.read_csv(filepath)
        df_hist['tourney_date'] = pd.to_datetime(df_hist['tourney_date'], format='%Y%m%d')
        df_hist = df_hist.sort_values(['tourney_date', 'match_num']).reset_index(drop=True)
        history_parts_40.append(df_hist[cols_base_40].dropna().copy())
    except FileNotFoundError:
        pass
df_history_40 = pd.concat(history_parts_40, ignore_index=True) if history_parts_40 else pd.DataFrame(columns=cols_base_40)

print(f"Historia — model 17-cechowy (2018–2023): {len(df_history_17)} meczów")
print(f"Historia — model 40-cechowy (2018–2023): {len(df_history_40)} meczów")

## 3. Label Encoding i podział chronologiczny

Nawierzchnia i poziom turnieju kodowane liczbowo (LabelEncoder). Enkodery dopasowywane na unii danych 2024 + historia, aby uniknąć nieznanych kategorii.

Podział danych 2024: **60% trening / 20% walidacja / 20% test** (chronologicznie).

In [ ]:
# --- Label Encoding: model 17-cechowy ---
le_surface_17 = LabelEncoder()
le_level_17 = LabelEncoder()
le_surface_17.fit(pd.concat([df_base_17['surface'], df_history_17['surface']]).unique())
le_level_17.fit(pd.concat([df_base_17['tourney_level'], df_history_17['tourney_level']]).unique())
df_base_17['surface_encoded'] = le_surface_17.transform(df_base_17['surface'])
df_base_17['tourney_level_encoded'] = le_level_17.transform(df_base_17['tourney_level'])

# --- Label Encoding: model 40-cechowy ---
le_surface_40 = LabelEncoder()
le_level_40 = LabelEncoder()
le_surface_40.fit(pd.concat([df_base_40['surface'], df_history_40['surface']]).unique())
le_level_40.fit(pd.concat([df_base_40['tourney_level'], df_history_40['tourney_level']]).unique())
df_base_40['surface_encoded'] = le_surface_40.transform(df_base_40['surface'])
df_base_40['tourney_level_encoded'] = le_level_40.transform(df_base_40['tourney_level'])

# --- Podział chronologiczny: model 17-cechowy ---
train_end_17 = int(len(df_base_17) * 0.60)
val_end_17 = int(len(df_base_17) * 0.80)
df_train_17 = df_base_17.iloc[:train_end_17].reset_index(drop=True)
df_val_17 = df_base_17.iloc[train_end_17:val_end_17].reset_index(drop=True)
df_test_17 = df_base_17.iloc[val_end_17:].reset_index(drop=True)
df_train_17['match_id'] = range(len(df_train_17))
df_val_17['match_id'] = range(len(df_val_17))
df_test_17['match_id'] = range(len(df_test_17))

# --- Podział chronologiczny: model 40-cechowy ---
train_end_40 = int(len(df_base_40) * 0.60)
val_end_40 = int(len(df_base_40) * 0.80)
df_train_40 = df_base_40.iloc[:train_end_40].reset_index(drop=True)
df_val_40 = df_base_40.iloc[train_end_40:val_end_40].reset_index(drop=True)
df_test_40 = df_base_40.iloc[val_end_40:].reset_index(drop=True)
df_train_40['match_id'] = range(len(df_train_40))
df_val_40['match_id'] = range(len(df_val_40))
df_test_40['match_id'] = range(len(df_test_40))

print(f"Model 17-cech — trening: {len(df_train_17)}, walidacja: {len(df_val_17)}, test: {len(df_test_17)}")
print(f"Model 40-cech — trening: {len(df_train_40)}, walidacja: {len(df_val_40)}, test: {len(df_test_40)}")

## 4. Funkcje cech dynamicznych

Funkcje współdzielone przez oba modele:
- **`calculate_form`** — wskaźnik zwycięstw z ostatnich 10 meczów (0.5 = brak historii)
- **`get_h2h`** — bilans bezpośrednich spotkań (p1 - p2)
- **`calculate_surface_form`** — forma na konkretnej nawierzchni (fallback do formy ogólnej przy < 3 meczach)

Dla modelu 40-cechowego dodatkowo:
- **`calculate_serve_stats`** — 8 statystyk serwisowych i returnowych (rolling avg z 10 meczów)

In [ ]:
# --- Funkcje wspólne ---
def calculate_form(player_name, history):
    player_history = history[(history['winner_name'] == player_name) |
                             (history['loser_name'] == player_name)].tail(10)
    if len(player_history) == 0:
        return 0.5
    wins = len(player_history[player_history['winner_name'] == player_name])
    return wins / len(player_history)


def get_h2h(p1, p2, history):
    p1_wins = len(history[(history['winner_name'] == p1) & (history['loser_name'] == p2)])
    p2_wins = len(history[(history['winner_name'] == p2) & (history['loser_name'] == p1)])
    return p1_wins - p2_wins


def calculate_surface_form(player_name, surface, history):
    surface_matches = history[history['surface'] == surface]
    player_on_surface = surface_matches[
        (surface_matches['winner_name'] == player_name) |
        (surface_matches['loser_name'] == player_name)
    ].tail(10)
    if len(player_on_surface) < 3:
        return calculate_form(player_name, history)
    wins = len(player_on_surface[player_on_surface['winner_name'] == player_name])
    return wins / len(player_on_surface)


# --- Statystyki serwisowe (tylko model 40-cechowy) ---
SERVE_STAT_NAMES = ['ace_rate', 'df_rate', 'first_in_pct', 'first_won_pct',
                    'second_won_pct', 'bp_save_pct', 'bp_faced_per_game', 'return_pts_won']

SERVE_DEFAULTS = {
    'ace_rate': 0.08, 'df_rate': 0.03, 'first_in_pct': 0.60,
    'first_won_pct': 0.70, 'second_won_pct': 0.50,
    'bp_save_pct': 0.60, 'bp_faced_per_game': 0.40, 'return_pts_won': 0.35
}


def calculate_serve_stats(player_name, history, window=10):
    player_matches = history[
        (history['winner_name'] == player_name) |
        (history['loser_name'] == player_name)
    ].tail(window)
    if len(player_matches) == 0:
        return SERVE_DEFAULTS.copy()

    ace_rates, df_rates, first_in_pcts = [], [], []
    first_won_pcts, second_won_pcts = [], []
    bp_save_pcts, bp_faced_per_games = [], []
    return_pts_won_pcts = []

    for _, match in player_matches.iterrows():
        is_winner = (match['winner_name'] == player_name)
        if is_winner:
            svpt, ace, df_ = match['w_svpt'], match['w_ace'], match['w_df']
            first_in, first_won = match['w_1stIn'], match['w_1stWon']
            second_won = match['w_2ndWon']
            sv_gms = match['w_SvGms']
            bp_saved, bp_faced = match['w_bpSaved'], match['w_bpFaced']
            opp_svpt = match['l_svpt']
            opp_first_won, opp_second_won = match['l_1stWon'], match['l_2ndWon']
        else:
            svpt, ace, df_ = match['l_svpt'], match['l_ace'], match['l_df']
            first_in, first_won = match['l_1stIn'], match['l_1stWon']
            second_won = match['l_2ndWon']
            sv_gms = match['l_SvGms']
            bp_saved, bp_faced = match['l_bpSaved'], match['l_bpFaced']
            opp_svpt = match['w_svpt']
            opp_first_won, opp_second_won = match['w_1stWon'], match['w_2ndWon']

        if svpt > 0:
            ace_rates.append(ace / svpt)
            df_rates.append(df_ / svpt)
            first_in_pcts.append(first_in / svpt)
        if first_in > 0:
            first_won_pcts.append(first_won / first_in)
        second_serve = svpt - first_in
        if second_serve > 0:
            second_won_pcts.append(second_won / second_serve)
        if bp_faced > 0:
            bp_save_pcts.append(bp_saved / bp_faced)
        if sv_gms > 0:
            bp_faced_per_games.append(bp_faced / sv_gms)
        if opp_svpt > 0:
            return_pts_won_pcts.append((opp_svpt - opp_first_won - opp_second_won) / opp_svpt)

    def safe_mean(lst, default):
        return np.mean(lst) if lst else default

    return {
        'ace_rate': safe_mean(ace_rates, SERVE_DEFAULTS['ace_rate']),
        'df_rate': safe_mean(df_rates, SERVE_DEFAULTS['df_rate']),
        'first_in_pct': safe_mean(first_in_pcts, SERVE_DEFAULTS['first_in_pct']),
        'first_won_pct': safe_mean(first_won_pcts, SERVE_DEFAULTS['first_won_pct']),
        'second_won_pct': safe_mean(second_won_pcts, SERVE_DEFAULTS['second_won_pct']),
        'bp_save_pct': safe_mean(bp_save_pcts, SERVE_DEFAULTS['bp_save_pct']),
        'bp_faced_per_game': safe_mean(bp_faced_per_games, SERVE_DEFAULTS['bp_faced_per_game']),
        'return_pts_won': safe_mean(return_pts_won_pcts, SERVE_DEFAULTS['return_pts_won']),
    }

print("Funkcje cech dynamicznych zdefiniowane.")

## 5. Obliczenie cech dynamicznych (jednorazowe)

Cechy dynamiczne nie zależą od ziarna losowości — obliczane są raz dla każdego modelu. To najdłuższy etap obliczeniowy (expanding window po każdym meczu).

- **Model 17-cechowy**: forma, H2H, forma nawierzchniowa
- **Model 40-cechowy**: jak wyżej + 8 statystyk serwisowych × 2 graczy

In [ ]:
# --- add_dynamic_features: model 17-cechowy (forma + H2H) ---
def add_dynamic_features_17(df_subset, historical_data):
    h2h_list, w_form_list, l_form_list = [], [], []
    w_sf_list, l_sf_list = [], []
    full_sequence = pd.concat([historical_data, df_subset]).reset_index(drop=True)
    start_idx = len(historical_data)
    for i in range(len(df_subset)):
        row = df_subset.iloc[i]
        past = full_sequence.iloc[:start_idx + i]
        p_w, p_l, surf = row['winner_name'], row['loser_name'], row['surface']
        h2h_list.append(get_h2h(p_w, p_l, past))
        w_form_list.append(calculate_form(p_w, past))
        l_form_list.append(calculate_form(p_l, past))
        w_sf_list.append(calculate_surface_form(p_w, surf, past))
        l_sf_list.append(calculate_surface_form(p_l, surf, past))
    df_subset = df_subset.copy()
    df_subset['h2h_diff'] = h2h_list
    df_subset['w_form'] = w_form_list
    df_subset['l_form'] = l_form_list
    df_subset['w_surface_form'] = w_sf_list
    df_subset['l_surface_form'] = l_sf_list
    return df_subset

print("Obliczanie cech dynamicznych — model 17-cechowy...")
t0 = time.time()
df_train_17 = add_dynamic_features_17(df_train_17, df_history_17)
hist_val_17 = pd.concat([df_history_17, df_train_17[cols_base_17]]).reset_index(drop=True)
df_val_17 = add_dynamic_features_17(df_val_17, hist_val_17)
hist_test_17 = pd.concat([df_history_17, df_train_17[cols_base_17], df_val_17[cols_base_17]]).reset_index(drop=True)
df_test_17 = add_dynamic_features_17(df_test_17, hist_test_17)
print(f"  Gotowe w {time.time()-t0:.1f}s")

In [ ]:
# --- add_dynamic_features: model 40-cechowy (forma + H2H + serve) ---
def add_dynamic_features_40(df_subset, historical_data):
    h2h_list, w_form_list, l_form_list = [], [], []
    w_sf_list, l_sf_list = [], []
    w_serve_stats, l_serve_stats = [], []
    full_sequence = pd.concat([historical_data, df_subset]).reset_index(drop=True)
    start_idx = len(historical_data)
    for i in range(len(df_subset)):
        row = df_subset.iloc[i]
        past = full_sequence.iloc[:start_idx + i]
        p_w, p_l, surf = row['winner_name'], row['loser_name'], row['surface']
        h2h_list.append(get_h2h(p_w, p_l, past))
        w_form_list.append(calculate_form(p_w, past))
        l_form_list.append(calculate_form(p_l, past))
        w_sf_list.append(calculate_surface_form(p_w, surf, past))
        l_sf_list.append(calculate_surface_form(p_l, surf, past))
        w_serve_stats.append(calculate_serve_stats(p_w, past))
        l_serve_stats.append(calculate_serve_stats(p_l, past))
    df_subset = df_subset.copy()
    df_subset['h2h_diff'] = h2h_list
    df_subset['w_form'] = w_form_list
    df_subset['l_form'] = l_form_list
    df_subset['w_surface_form'] = w_sf_list
    df_subset['l_surface_form'] = l_sf_list
    for stat in SERVE_STAT_NAMES:
        df_subset[f'w_{stat}'] = [s[stat] for s in w_serve_stats]
        df_subset[f'l_{stat}'] = [s[stat] for s in l_serve_stats]
    return df_subset

print("Obliczanie cech dynamicznych — model 40-cechowy...")
t0 = time.time()
df_train_40 = add_dynamic_features_40(df_train_40, df_history_40)
hist_val_40 = pd.concat([df_history_40, df_train_40[cols_base_40]]).reset_index(drop=True)
df_val_40 = add_dynamic_features_40(df_val_40, hist_val_40)
hist_test_40 = pd.concat([df_history_40, df_train_40[cols_base_40], df_val_40[cols_base_40]]).reset_index(drop=True)
df_test_40 = add_dynamic_features_40(df_test_40, hist_test_40)
print(f"  Gotowe w {time.time()-t0:.1f}s")

## 6. Symetryzacja danych

Etykieta `player_won = 1` oznacza, że gracz P1 wygrał. Aby model nie faworyzował kolejności graczy, dane symetryzujemy — każdy mecz dodawany w obu orientacjach (P1↔P2), z odwróconym wynikiem.

### Model 17-cechowy
Symetryzowane cechy: forma, forma nawierzchniowa, ranking (log), is_lefty, H2H, surface, tourney_level.

### Model 40-cechowy
Jak wyżej + 8 statystyk serwisowych × 2, best_of, round_encoded, rank_pts_log.

In [ ]:
# --- Symetryzacja: model 17-cechowy ---
def symmetrize_data_17(df_subset, seed, shuffle=True):
    rows_p1_wins, rows_p2_wins = [], []
    for idx, row in df_subset.iterrows():
        row1 = {
            'match_id': row['match_id'],
            'surface': row['surface_encoded'], 'tourney_level': row['tourney_level_encoded'],
            'p1_rank_log': row['winner_rank_log'], 'p1_age': row['winner_age'], 'p1_ht': row['winner_ht'],
            'p2_rank_log': row['loser_rank_log'], 'p2_age': row['loser_age'], 'p2_ht': row['loser_ht'],
            'p1_h2h': row['h2h_diff'],
            'p1_form': row['w_form'], 'p2_form': row['l_form'],
            'p1_surface_form': row['w_surface_form'], 'p2_surface_form': row['l_surface_form'],
            'rank_diff': row['winner_rank_log'] - row['loser_rank_log'],
            'age_diff': row['winner_age'] - row['loser_age'],
            'ht_diff': row['winner_ht'] - row['loser_ht'],
            'form_diff': row['w_form'] - row['l_form'],
            'y': 1, 'actual_winner': row['winner_name'], 'actual_loser': row['loser_name'],
            'p1_name': row['winner_name'], 'p2_name': row['loser_name']
        }
        row2 = {
            'match_id': row['match_id'],
            'surface': row['surface_encoded'], 'tourney_level': row['tourney_level_encoded'],
            'p1_rank_log': row['loser_rank_log'], 'p1_age': row['loser_age'], 'p1_ht': row['loser_ht'],
            'p2_rank_log': row['winner_rank_log'], 'p2_age': row['winner_age'], 'p2_ht': row['winner_ht'],
            'p1_h2h': -row['h2h_diff'],
            'p1_form': row['l_form'], 'p2_form': row['w_form'],
            'p1_surface_form': row['l_surface_form'], 'p2_surface_form': row['w_surface_form'],
            'rank_diff': row['loser_rank_log'] - row['winner_rank_log'],
            'age_diff': row['loser_age'] - row['winner_age'],
            'ht_diff': row['loser_ht'] - row['winner_ht'],
            'form_diff': row['l_form'] - row['w_form'],
            'y': 0, 'actual_winner': row['winner_name'], 'actual_loser': row['loser_name'],
            'p1_name': row['loser_name'], 'p2_name': row['winner_name']
        }
        rows_p1_wins.append(row1)
        rows_p2_wins.append(row2)

    all_rows = []
    for r1, r2 in zip(rows_p1_wins, rows_p2_wins):
        all_rows.extend([r1, r2])
    result = pd.DataFrame(all_rows)
    if shuffle:
        result = result.sample(frac=1, random_state=seed).reset_index(drop=True)
    else:
        result = result.reset_index(drop=True)
    return result

print("Funkcja symetryzacji 17-cechowej zdefiniowana.")

In [ ]:
# --- Symetryzacja: model 40-cechowy ---
def symmetrize_data_40(df_subset, seed, shuffle=True):
    rows_p1_wins, rows_p2_wins = [], []
    for idx, row in df_subset.iterrows():
        row1 = {
            'match_id': row['match_id'],
            'surface': row['surface_encoded'], 'tourney_level': row['tourney_level_encoded'],
            'best_of': row['best_of'], 'round_num': row['round_encoded'],
            'p1_rank_log': row['winner_rank_log'], 'p1_rank_pts_log': row['winner_rank_pts_log'],
            'p1_age': row['winner_age'], 'p1_ht': row['winner_ht'], 'p1_is_lefty': row['winner_is_lefty'],
            'p2_rank_log': row['loser_rank_log'], 'p2_rank_pts_log': row['loser_rank_pts_log'],
            'p2_age': row['loser_age'], 'p2_ht': row['loser_ht'], 'p2_is_lefty': row['loser_is_lefty'],
            'p1_h2h': row['h2h_diff'],
            'p1_form': row['w_form'], 'p2_form': row['l_form'],
            'p1_surface_form': row['w_surface_form'], 'p2_surface_form': row['l_surface_form'],
            'rank_diff': row['winner_rank_log'] - row['loser_rank_log'],
            'rank_pts_diff': row['winner_rank_pts_log'] - row['loser_rank_pts_log'],
            'age_diff': row['winner_age'] - row['loser_age'],
            'ht_diff': row['winner_ht'] - row['loser_ht'],
            'form_diff': row['w_form'] - row['l_form'],
            'y': 1, 'actual_winner': row['winner_name'], 'actual_loser': row['loser_name'],
            'p1_name': row['winner_name'], 'p2_name': row['loser_name']
        }
        for s in SERVE_STAT_NAMES:
            row1[f'p1_{s}'] = row[f'w_{s}']
            row1[f'p2_{s}'] = row[f'l_{s}']

        row2 = {
            'match_id': row['match_id'],
            'surface': row['surface_encoded'], 'tourney_level': row['tourney_level_encoded'],
            'best_of': row['best_of'], 'round_num': row['round_encoded'],
            'p1_rank_log': row['loser_rank_log'], 'p1_rank_pts_log': row['loser_rank_pts_log'],
            'p1_age': row['loser_age'], 'p1_ht': row['loser_ht'], 'p1_is_lefty': row['loser_is_lefty'],
            'p2_rank_log': row['winner_rank_log'], 'p2_rank_pts_log': row['winner_rank_pts_log'],
            'p2_age': row['winner_age'], 'p2_ht': row['winner_ht'], 'p2_is_lefty': row['winner_is_lefty'],
            'p1_h2h': -row['h2h_diff'],
            'p1_form': row['l_form'], 'p2_form': row['w_form'],
            'p1_surface_form': row['l_surface_form'], 'p2_surface_form': row['w_surface_form'],
            'rank_diff': row['loser_rank_log'] - row['winner_rank_log'],
            'rank_pts_diff': row['loser_rank_pts_log'] - row['winner_rank_pts_log'],
            'age_diff': row['loser_age'] - row['winner_age'],
            'ht_diff': row['loser_ht'] - row['winner_ht'],
            'form_diff': row['l_form'] - row['w_form'],
            'y': 0, 'actual_winner': row['winner_name'], 'actual_loser': row['loser_name'],
            'p1_name': row['loser_name'], 'p2_name': row['winner_name']
        }
        for s in SERVE_STAT_NAMES:
            row2[f'p1_{s}'] = row[f'l_{s}']
            row2[f'p2_{s}'] = row[f'w_{s}']

        rows_p1_wins.append(row1)
        rows_p2_wins.append(row2)

    all_rows = []
    for r1, r2 in zip(rows_p1_wins, rows_p2_wins):
        all_rows.extend([r1, r2])
    result = pd.DataFrame(all_rows)
    if shuffle:
        result = result.sample(frac=1, random_state=seed).reset_index(drop=True)
    else:
        result = result.reset_index(drop=True)
    return result

print("Funkcja symetryzacji 40-cechowej zdefiniowana.")

## 7. Listy cech i hiperparametry

**Model 17-cechowy** (17 features):
`surface, tourney_level, p1/p2_rank_log, p1/p2_age, p1/p2_ht, p1_h2h, p1/p2_form, p1/p2_surface_form, rank_diff, age_diff, ht_diff, form_diff`

**Model 40-cechowy** (40 features):
Jak wyżej + `best_of, round_num, p1/p2_rank_pts_log, p1/p2_is_lefty, 8 × p1/p2_serve_stats, rank_pts_diff`

Przestrzeń przeszukiwania hiperparametrów (`RandomizedSearchCV`, `n_iter=50`, `TimeSeriesSplit(5)`) jest identyczna dla obu modeli.

In [ ]:
# --- Lista cech: model 17-cechowy ---
features_17 = ['surface', 'tourney_level', 'p1_rank_log', 'p1_age', 'p1_ht',
               'p2_rank_log', 'p2_age', 'p2_ht', 'p1_h2h', 'p1_form', 'p2_form',
               'p1_surface_form', 'p2_surface_form',
               'rank_diff', 'age_diff', 'ht_diff', 'form_diff']

# --- Lista cech: model 40-cechowy ---
features_40 = ['surface', 'tourney_level', 'best_of', 'round_num',
               'p1_rank_log', 'p1_rank_pts_log', 'p1_age', 'p1_ht', 'p1_is_lefty',
               'p2_rank_log', 'p2_rank_pts_log', 'p2_age', 'p2_ht', 'p2_is_lefty',
               'p1_h2h', 'p1_form', 'p2_form',
               'p1_surface_form', 'p2_surface_form',
               'p1_ace_rate', 'p2_ace_rate', 'p1_df_rate', 'p2_df_rate',
               'p1_first_in_pct', 'p2_first_in_pct',
               'p1_first_won_pct', 'p2_first_won_pct',
               'p1_second_won_pct', 'p2_second_won_pct',
               'p1_bp_save_pct', 'p2_bp_save_pct',
               'p1_bp_faced_per_game', 'p2_bp_faced_per_game',
               'p1_return_pts_won', 'p2_return_pts_won',
               'rank_diff', 'rank_pts_diff', 'age_diff', 'ht_diff', 'form_diff']

# --- Hiperparametry (wspólne) ---
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 15, 20, 30, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True],
    'max_samples': [0.7, 0.8, 0.9, 1.0]
}

print(f"Cechy — model 17: {len(features_17)}")
print(f"Cechy — model 40: {len(features_40)}")

## 8. Pętla multi-seed — model 17-cechowy

Dla każdego z 10 ziaren (1–10):
1. Symetryzacja danych (train/val/test) z danym ziarnem
2. `RandomizedSearchCV` (50 iteracji, `TimeSeriesSplit(5)`) z tym ziarnem
3. Trening finalnego RF na pełnym zbiorze treningowym
4. Ewaluacja: CV Accuracy, Validation Accuracy, Test Accuracy, **Match Accuracy**

In [ ]:
results_17 = []

print(f"\n{'='*60}")
print(f"  MODEL 17-CECHOWY — TEST {N_RUNS} URUCHOMIEŃ")
print(f"{'='*60}\n")

for run_idx, seed in enumerate(SEEDS, 1):
    t_start = time.time()
    print(f"--- Przebieg {run_idx}/{N_RUNS} (seed={seed}) ---")

    # Symetryzacja z bieżącym ziarnem
    val_data = symmetrize_data_17(df_val_17, seed, shuffle=True)
    test_data = symmetrize_data_17(df_test_17, seed, shuffle=True)
    train_data_ordered = symmetrize_data_17(df_train_17, seed, shuffle=False)
    train_data_final = symmetrize_data_17(df_train_17, seed, shuffle=True)

    X_train_cv = train_data_ordered[features_17]
    y_train_cv = train_data_ordered['y']
    X_val = val_data[features_17]
    y_val = val_data['y']
    X_test = test_data[features_17]
    y_test = test_data['y']

    # RandomizedSearchCV
    rf = RandomForestClassifier(n_jobs=1, random_state=seed)
    tscv = TimeSeriesSplit(n_splits=5)
    search = RandomizedSearchCV(rf, param_dist, n_iter=50, cv=tscv,
                                scoring='accuracy', n_jobs=-1, verbose=0,
                                random_state=seed)
    search.fit(X_train_cv, y_train_cv)

    best_rf = search.best_estimator_
    best_rf.n_jobs = -1

    # Trening finalny
    X_train_final = train_data_final[features_17]
    y_train_final = train_data_final['y']
    best_rf.fit(X_train_final, y_train_final)

    # Ewaluacja
    val_acc = accuracy_score(y_val, best_rf.predict(X_val))
    test_pred_proba = best_rf.predict_proba(X_test)
    test_acc = accuracy_score(y_test, best_rf.predict(X_test))

    # Match accuracy
    test_data_copy = test_data.copy()
    test_data_copy['p1_win_prob'] = test_pred_proba[:, 1]
    wp = test_data_copy[test_data_copy['y'] == 1].copy()
    wp['correct'] = wp['p1_win_prob'] > 0.5
    match_acc = wp['correct'].mean()

    elapsed = time.time() - t_start
    results_17.append({
        'seed': seed,
        'cv_acc': search.best_score_,
        'val_acc': val_acc,
        'test_acc': test_acc,
        'match_acc': match_acc,
        'best_params': search.best_params_,
        'time_s': elapsed
    })
    print(f"  CV={search.best_score_:.4f}  Val={val_acc:.4f}  Test={test_acc:.4f}  "
          f"Match={match_acc:.4f} ({match_acc*100:.2f}%)  [{elapsed:.1f}s]")

print(f"\nModel 17-cechowy zakończony.")

## 9. Pętla multi-seed — model 40-cechowy

Identyczna procedura jak dla modelu 17-cechowego, ale z rozszerzonym zestawem 40 cech i statystykami serwisowymi.

In [ ]:
results_40 = []

print(f"\n{'='*60}")
print(f"  MODEL 40-CECHOWY — TEST {N_RUNS} URUCHOMIEŃ")
print(f"{'='*60}\n")

for run_idx, seed in enumerate(SEEDS, 1):
    t_start = time.time()
    print(f"--- Przebieg {run_idx}/{N_RUNS} (seed={seed}) ---")

    # Symetryzacja z bieżącym ziarnem
    val_data = symmetrize_data_40(df_val_40, seed, shuffle=True)
    test_data = symmetrize_data_40(df_test_40, seed, shuffle=True)
    train_data_ordered = symmetrize_data_40(df_train_40, seed, shuffle=False)
    train_data_final = symmetrize_data_40(df_train_40, seed, shuffle=True)

    X_train_cv = train_data_ordered[features_40]
    y_train_cv = train_data_ordered['y']
    X_val = val_data[features_40]
    y_val = val_data['y']
    X_test = test_data[features_40]
    y_test = test_data['y']

    # RandomizedSearchCV
    rf = RandomForestClassifier(n_jobs=1, random_state=seed)
    tscv = TimeSeriesSplit(n_splits=5)
    search = RandomizedSearchCV(rf, param_dist, n_iter=50, cv=tscv,
                                scoring='accuracy', n_jobs=-1, verbose=0,
                                random_state=seed)
    search.fit(X_train_cv, y_train_cv)

    best_rf = search.best_estimator_
    best_rf.n_jobs = -1

    # Trening finalny
    X_train_final = train_data_final[features_40]
    y_train_final = train_data_final['y']
    best_rf.fit(X_train_final, y_train_final)

    # Ewaluacja
    val_acc = accuracy_score(y_val, best_rf.predict(X_val))
    test_pred_proba = best_rf.predict_proba(X_test)
    test_acc = accuracy_score(y_test, best_rf.predict(X_test))

    # Match accuracy
    test_data_copy = test_data.copy()
    test_data_copy['p1_win_prob'] = test_pred_proba[:, 1]
    wp = test_data_copy[test_data_copy['y'] == 1].copy()
    wp['correct'] = wp['p1_win_prob'] > 0.5
    match_acc = wp['correct'].mean()

    elapsed = time.time() - t_start
    results_40.append({
        'seed': seed,
        'cv_acc': search.best_score_,
        'val_acc': val_acc,
        'test_acc': test_acc,
        'match_acc': match_acc,
        'best_params': search.best_params_,
        'time_s': elapsed
    })
    print(f"  CV={search.best_score_:.4f}  Val={val_acc:.4f}  Test={test_acc:.4f}  "
          f"Match={match_acc:.4f} ({match_acc*100:.2f}%)  [{elapsed:.1f}s]")

print(f"\nModel 40-cechowy zakończony.")

## 10. Tabela wyników porównawczych

Zestawienie wyników obu modeli w formie tabel z metrykami:
- **CV Accuracy** — średnia z cross-validation
- **Validation Accuracy** — na zbiorze walidacyjnym
- **Test Accuracy** — na zbiorze testowym (symetryzowane pary)
- **Match Accuracy** — prawdziwa trafność meczowa (czy model poprawnie wskazał zwycięzcę)

Statystyki: średnia, odchylenie standardowe, minimum, maksimum.

In [ ]:
# --- Wyniki: model 17-cechowy ---
print(f"\n{'='*60}")
print(f"  MODEL 17-CECHOWY — PODSUMOWANIE ({N_RUNS} uruchomień)")
print(f"{'='*60}\n")

cv_17 = [r['cv_acc'] for r in results_17]
val_17 = [r['val_acc'] for r in results_17]
test_17 = [r['test_acc'] for r in results_17]
match_17 = [r['match_acc'] for r in results_17]

print(f"{'Seed':>6} {'CV Acc':>10} {'Val Acc':>10} {'Test Acc':>10} {'Match Acc':>12}")
print("-" * 52)
for r in results_17:
    print(f"{r['seed']:>6} {r['cv_acc']:>10.4f} {r['val_acc']:>10.4f} "
          f"{r['test_acc']:>10.4f} {r['match_acc']:>10.4f} ({r['match_acc']*100:.2f}%)")
print("-" * 52)
print(f"{'ŚREDNIA':>6} {np.mean(cv_17):>10.4f} {np.mean(val_17):>10.4f} "
      f"{np.mean(test_17):>10.4f} {np.mean(match_17):>10.4f} ({np.mean(match_17)*100:.2f}%)")
print(f"{'STD':>6} {np.std(cv_17):>10.4f} {np.std(val_17):>10.4f} "
      f"{np.std(test_17):>10.4f} {np.std(match_17):>10.4f}")
print(f"{'MIN':>6} {np.min(cv_17):>10.4f} {np.min(val_17):>10.4f} "
      f"{np.min(test_17):>10.4f} {np.min(match_17):>10.4f} ({np.min(match_17)*100:.2f}%)")
print(f"{'MAX':>6} {np.max(cv_17):>10.4f} {np.max(val_17):>10.4f} "
      f"{np.max(test_17):>10.4f} {np.max(match_17):>10.4f} ({np.max(match_17)*100:.2f}%)")

# --- Wyniki: model 40-cechowy ---
print(f"\n{'='*60}")
print(f"  MODEL 40-CECHOWY — PODSUMOWANIE ({N_RUNS} uruchomień)")
print(f"{'='*60}\n")

cv_40 = [r['cv_acc'] for r in results_40]
val_40 = [r['val_acc'] for r in results_40]
test_40 = [r['test_acc'] for r in results_40]
match_40 = [r['match_acc'] for r in results_40]

print(f"{'Seed':>6} {'CV Acc':>10} {'Val Acc':>10} {'Test Acc':>10} {'Match Acc':>12}")
print("-" * 52)
for r in results_40:
    print(f"{r['seed']:>6} {r['cv_acc']:>10.4f} {r['val_acc']:>10.4f} "
          f"{r['test_acc']:>10.4f} {r['match_acc']:>10.4f} ({r['match_acc']*100:.2f}%)")
print("-" * 52)
print(f"{'ŚREDNIA':>6} {np.mean(cv_40):>10.4f} {np.mean(val_40):>10.4f} "
      f"{np.mean(test_40):>10.4f} {np.mean(match_40):>10.4f} ({np.mean(match_40)*100:.2f}%)")
print(f"{'STD':>6} {np.std(cv_40):>10.4f} {np.std(val_40):>10.4f} "
      f"{np.std(test_40):>10.4f} {np.std(match_40):>10.4f}")
print(f"{'MIN':>6} {np.min(cv_40):>10.4f} {np.min(val_40):>10.4f} "
      f"{np.min(test_40):>10.4f} {np.min(match_40):>10.4f} ({np.min(match_40)*100:.2f}%)")
print(f"{'MAX':>6} {np.max(cv_40):>10.4f} {np.max(val_40):>10.4f} "
      f"{np.max(test_40):>10.4f} {np.max(match_40):>10.4f} ({np.max(match_40)*100:.2f}%)")

## 11. Wizualizacja: Match Accuracy per Seed

Wykres słupkowy porównujący Match Accuracy obu modeli dla każdego z 10 ziaren losowości. Linia przerywana pokazuje średnią każdego modelu.

In [ ]:
import matplotlib.pyplot as plt

x = np.arange(1, N_RUNS + 1)
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, [r['match_acc'] * 100 for r in results_17], width,
               label=f'Model 17-cech (śr. {np.mean(match_17)*100:.2f}%)', color='#4C72B0', alpha=0.85)
bars2 = ax.bar(x + width/2, [r['match_acc'] * 100 for r in results_40], width,
               label=f'Model 40-cech (śr. {np.mean(match_40)*100:.2f}%)', color='#DD8452', alpha=0.85)

ax.axhline(y=np.mean(match_17) * 100, color='#4C72B0', linestyle='--', linewidth=1, alpha=0.7)
ax.axhline(y=np.mean(match_40) * 100, color='#DD8452', linestyle='--', linewidth=1, alpha=0.7)

ax.set_xlabel('Seed (ziarno losowości)', fontsize=12)
ax.set_ylabel('Match Accuracy (%)', fontsize=12)
ax.set_title('Porównanie stabilności: Model 17-cech vs 40-cech (10 uruchomień)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in SEEDS])
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Wartości nad słupkami
for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.15, f'{h:.1f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.15, f'{h:.1f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## 12. Podsumowanie i wnioski

### Porównanie stabilności modeli

| Metryka | Model 17-cech | Model 40-cech |
|---------|:------------:|:-------------:|
| Liczba cech | 17 | 40 |
| Historia (cold-start) | 2018–2023 (6 lat) | 2018–2023 (6 lat) |
| Statystyki serwisowe | ❌ | ✅ |
| Dodatkowe cechy | — | best_of, round, hand, rank_pts |

**Cel eksperymentu**: Sprawdzenie, czy rozszerzenie modelu o dodatkowe cechy (serwis, rank_pts, ręczność, runda, best_of) prowadzi do **stabilnej** poprawy trafności predykcji meczów tenisowych ATP.

**Kluczowa metryka**: Match Accuracy — odsetek meczów, dla których model poprawnie wskazał zwycięzcę.

Niskie odchylenie standardowe (STD) świadczy o stabilności modelu niezależnie od ziarna losowości.

In [ ]:
# --- Podsumowanie końcowe ---
print(f"\n{'='*65}")
print(f"  PORÓWNANIE KOŃCOWE")
print(f"{'='*65}")
print(f"\n{'':>20} {'Model 17-cech':>15} {'Model 40-cech':>15}")
print(f"{'-'*52}")
print(f"{'Śr. Match Acc':>20} {np.mean(match_17)*100:>14.2f}% {np.mean(match_40)*100:>14.2f}%")
print(f"{'STD Match Acc':>20} {np.std(match_17)*100:>14.2f}% {np.std(match_40)*100:>14.2f}%")
print(f"{'Min Match Acc':>20} {np.min(match_17)*100:>14.2f}% {np.min(match_40)*100:>14.2f}%")
print(f"{'Max Match Acc':>20} {np.max(match_17)*100:>14.2f}% {np.max(match_40)*100:>14.2f}%")
print(f"{'Śr. CV Acc':>20} {np.mean(cv_17)*100:>14.2f}% {np.mean(cv_40)*100:>14.2f}%")
print(f"{'Śr. Val Acc':>20} {np.mean(val_17)*100:>14.2f}% {np.mean(val_40)*100:>14.2f}%")
print(f"{'Śr. Test Acc':>20} {np.mean(test_17)*100:>14.2f}% {np.mean(test_40)*100:>14.2f}%")

total_17 = sum(r['time_s'] for r in results_17)
total_40 = sum(r['time_s'] for r in results_40)
print(f"\n{'Łączny czas':>20} {total_17:>13.1f}s {total_40:>13.1f}s")
print(f"{'Śr. czas/przebieg':>20} {total_17/N_RUNS:>13.1f}s {total_40/N_RUNS:>13.1f}s")